# C02Shift p10 gate\n\nThis notebook runs the shortest honest path for the new method. It trains the C02Shift synthetic prototype and then checks the direct `p10` field anchor before spending time on the full temporal benchmark.

In [ ]:
from google.colab import drive\nfrom pathlib import Path\nimport os\nimport sys\n\ndrive.mount('/content/drive')\n\nWORKSPACE = Path('/content/drive/MyDrive/Propagation')\nif not WORKSPACE.exists():\n    raise FileNotFoundError(f'Workspace not found: {WORKSPACE}')\n\nos.chdir(WORKSPACE)\nprint('Workspace:', WORKSPACE)\nprint('Python:', sys.version)\nif sys.version_info < (3, 11):\n    raise RuntimeError('This repo currently requires Python 3.11 or newer.')

In [ ]:
required_paths = [\n    Path('examples/sleipner_manifest.p10_11inline.json'),\n    Path('examples/exports/raw/94p10mid.sgy'),\n    Path('examples/exports/raw/10p10mid.sgy')\n]\nmissing = [str(path) for path in required_paths if not path.exists()]\nif missing:\n    raise FileNotFoundError('Missing required p10 benchmark files:\n' + '\n'.join(missing))\nprint('Required p10 benchmark files found.')

In [ ]:
!python -m pip install -U pip setuptools wheel\n!python -m pip install -e .[viz]

In [ ]:
import torch\nprint('CUDA available:', torch.cuda.is_available())\nif torch.cuda.is_available():\n    print('GPU:', torch.cuda.get_device_name(0))

In [ ]:
!PYTHONPATH=src python -m ccs_monitoring.cli run-all --config configs/paper_wave_temporal_colab.yaml

In [ ]:
!PYTHONPATH=src python -m ccs_monitoring.cli evaluate-field --config configs/sleipner_p10_wave_temporal_colab.yaml

In [ ]:
import json\nfrom pathlib import Path\n\nbaseline_path = Path('runs/sleipner_p10_11inline/results/summary.json')\nwave_path = Path('runs/sleipner_p10_wave_temporal_colab/results/summary.json')\n\nif not baseline_path.exists():\n    raise FileNotFoundError('Baseline p10 summary is missing. Expected an existing local baseline run at ' + str(baseline_path))\nif not wave_path.exists():\n    raise FileNotFoundError('C02Shift p10 summary is missing at ' + str(wave_path))\n\nbaseline = json.loads(baseline_path.read_text())\nwave = json.loads(wave_path.read_text())\n\nbaseline_metrics = baseline['Field']['volume_summary']['overall']['plain_ml_structured_constrained']\nwave_metrics = wave['Field']['volume_summary']['overall'].get('wave_temporal_constrained', {})\n\nprint('Baseline structured plain support-volume IoU:', baseline_metrics.get('support_volume_iou_2010'))\nprint('C02Shift support-volume IoU:', wave_metrics.get('support_volume_iou_2010'))\nprint('Baseline structured plain crossline continuity:', baseline_metrics.get('crossline_continuity'))\nprint('C02Shift crossline continuity:', wave_metrics.get('crossline_continuity'))\nprint('Baseline structured plain outside-support fraction:', baseline_metrics.get('predicted_fraction_outside_support_volume'))\nprint('C02Shift outside-support fraction:', wave_metrics.get('predicted_fraction_outside_support_volume'))\n\ngate_pass = (wave_metrics.get('support_volume_iou_2010', float('-inf')) > baseline_metrics.get('support_volume_iou_2010', float('inf')))
print('Direct p10 novelty gate passed:', gate_pass)